In [0]:
#Create the job
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import (
    Task, NotebookTask, Source, TaskDependency, CronSchedule, PauseStatus
)

w = WorkspaceClient()
ME = w.current_user.me().user_name
BASE = f"/Workspace/Users/{ME}/supplyChain-Data-Engineering"
JOB_NAME = "Supply Chain ETL Pipeline"

# task name: (notebook, tasks that must finish first)
pipeline = {
    "bronze_csv":            ("02_bronze_csv", []),
    "bronze_tsv":            ("03_bronze_tsv", []),
    "bronze_excel":          ("04_bronze_excel", []),
    "bronze_json":           ("05_bronze_json", []),
    "bronze_invoices":       ("06_bronze_invoices", []),
    "bronze_images_logs":    ("07_bronze_images_logs", []),
    "silver_core":           ("08_silver_core_tables", ["bronze_csv", "bronze_tsv", "bronze_excel"]),
    "silver_invoices":       ("09_silver_invoices", ["bronze_invoices", "silver_core"]),
    "silver_quality":        ("12_silver_quality_inspections", ["bronze_excel"]),
    "gold_dimensions":       ("10_gold_dimensions", ["silver_core"]),
    "gold_facts_aggregates": ("11_gold_facts_aggregates", ["gold_dimensions"]),
    "gold_kpi_snapshot":     ("13_gold_kpi_snapshot", ["gold_facts_aggregates", "silver_quality"]),
    "genie_comments":        ("14_genie_table_comments", ["gold_kpi_snapshot"]),
}

tasks = [
    Task(
        task_key=key,
        notebook_task=NotebookTask(notebook_path=f"{BASE}/{notebook}", source=Source.WORKSPACE),
        depends_on=[TaskDependency(task_key=d) for d in deps],
        max_retries=1,
        min_retry_interval_millis=60000,   # wait 1 minute before the retry
    )
    for key, (notebook, deps) in pipeline.items()
]

existing = list(w.jobs.list(name=JOB_NAME))
if existing:
    print("Job already exists, not creating a second one. Job ID:", existing[0].job_id)
else:
    job = w.jobs.create(
        name=JOB_NAME,
        tasks=tasks,
        max_concurrent_runs=1,
        schedule=CronSchedule(
            quartz_cron_expression="0 0 6 * * ?",   # every day at 06:00
            timezone_id="Asia/Kolkata",
            pause_status=PauseStatus.PAUSED,        # exists, but does not run by itself
        ),
    )
    print("✅ Created job. Job ID:", job.job_id)

In [0]:
#Checking that the run worked
C = "capstone_supply_chain"

# 1. The KPI snapshot should have a fresh timestamp
spark.table(f"{C}.gold_consolidated.supply_chain_kpi_snapshot") \
     .select("snapshot_timestamp", "total_purchase_orders", "on_time_delivery_pct").show(truncate=False)

# 2. The Genie comments must still be there (should be 8 tables)
print("tables with comments:", spark.sql(f"""
    SELECT COUNT(*) FROM {C}.information_schema.tables
    WHERE table_schema IN ('gold', 'gold_consolidated') AND comment IS NOT NULL
""").first()[0])